# Important Hyper-Parameters for QLoRA and Fine-Tuning

## QLoRA Hyperparameters:

When fine-tuning an LLM using QLoRA, you are not just pressing "Train." You are configuring a highly precise architecture. Understanding these five hyperparameters is the difference between a model that successfully learns your task and a model that crashes your GPU with an Out-of-Memory (OOM) error.

### 1. Quantization (The Base Model Compression):
- What it is: Quantization is the process of compressing the massive pre-trained Base Model (like Llama 3) before you attach your training adapters. Instead of loading the model in full 16-bit precision, you force it into a smaller data type.

- How it works in QLoRA: QLoRA specifically relies on NF4 (NormalFloat4) quantization. It assumes the neural network's weights follow a bell curve, and it mathematically squishes those 16-bit decimals down into just 16 possible values (4-bit).

- How it affects Fine-Tuning:
    - VRAM: This is what makes local training possible. It shrinks a 16GB model down to about 5GB.
    - Performance: The 4-bit base model is completely frozen during training. It acts only as a massive, read-only calculator. You lose a microscopic fraction of accuracy, but gain massive memory efficiency.

### 2. r (Rank):
- What it is: The core of LoRA. r represents the size (or "rank") of the bottleneck in the tiny adapter matrices you are attaching to the model.

- How it works: Instead of updating a 10,000 x 10,000 matrix, LoRA trains two smaller matrices: a 10,000 x r matrix, and an r x 10,000 matrix.

- How it affects Fine-Tuning:

    - Low Rank (r = 8 or 16): Perfect for teaching the model a specific tone, a conversational style, or strict output formatting (like generating JSON). Fast to train, very low VRAM cost.

    - High Rank (r = 32, 64, or 128): Required if you are forcing the model to learn entirely new, complex knowledge (like a foreign language or proprietary company code).

    - The Trade-off: Doubling your r doubles the number of trainable parameters. This increases training time, increases VRAM usage, and increases the risk of overfitting.

### 3. lora_alpha (The Scaling Factor):
- What it is: Think of alpha as the volume knob for your new LoRA adapters.

- How it works: When the model generates text, it combines the knowledge from the frozen 4-bit base model with the new knowledge from your trained adapters. alpha mathematically scales how heavily the model should weigh your new adapter weights versus the original weights.

- How it affects Fine-Tuning:

    - If alpha is too low, the model ignores your fine-tuning and acts like the base model.

    - If alpha is too high, the model's outputs become corrupted, repetitive, or completely nonsensical.

    - The Industry Standard Rule: You should generally set alpha to be exactly twice the value of your r.

        - If r = 16, set lora_alpha = 32.

        - If r = 64, set lora_alpha = 128.

### 4. target_modules:
- What it is: A neural network is made of many different components. target_modules is a list where you explicitly tell the code exactly which layers of the base model should get a LoRA adapter attached to them.

- How it works: In modern LLMs (Transformers), the most critical layers are the "Attention" mechanisms. The standard targets are the Query Projections (q_proj) and Value Projections (v_proj).

- How it affects Fine-Tuning:

    - Standard Target (["q_proj", "v_proj"]): Highly efficient. Gives great results for standard conversational fine-tuning while using very little VRAM.

    - Aggressive Target (["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]): Attaching adapters to every linear layer in the network. This yields the highest possible accuracy and helps the model learn faster, but it significantly spikes your VRAM usage.

### 5. lora_dropout:
- What it is: A regularization technique designed purely to stop the model from cheating.

- How it works: During the training loop, lora_dropout randomly turns off a specific percentage of the neurons in your LoRA adapter.

- How it affects Fine-Tuning:

    - Preventing Overfitting: If a model has too much capacity (like a high r value) on a small dataset, it will just memorize the exact answers instead of learning the underlying logic. By constantly turning off random neurons, Dropout forces the network to distribute the knowledge evenly.

    - Standard Values: Usually kept very low for LLMs.

        - 0.05 (5%) or 0.10 (10%) is the golden standard.

        - Setting it to 0.0 means no dropout, which is risky on tiny datasets. Setting it too high (like 0.5) will prevent the model from learning anything at all.

## Training Hyperparameters:
While QLoRA dictates what you are training, the Training Arguments dictate how the model actually learns. When working with consumer GPUs, managing these specific settings is how you prevent catastrophic Out-Of-Memory (OOM) crashes while still getting enterprise-grade results.

### 1. per_device_train_batch_size:
- What it is: Batch size is how many examples (prompts) the model looks at simultaneously before pausing to calculate its error and update its weights.

- How it affects Fine-Tuning:

    - Large Batch Size (e.g., 16, 32, 64): The model gets a very stable, accurate view of the data before updating. However, it requires holding 16 or 32 massive text sequences in your VRAM all at once.

    - Small Batch Size (e.g., 1, 2, or 4): Easily fits in memory, but the learning becomes very "noisy" and erratic because the model is updating its entire worldview based on just 1 or 2 examples at a time.

    - The Reality: On local hardware, you almost always have to set this to 1, 2, or 4.

### 2. gradient_accumulation_steps:
- What it is:This is the ultimate workaround for small VRAM. It allows you to simulate a massive batch size without actually needing the memory for it.
- How it works:If your VRAM can only handle a Batch Size of 2, but you want the mathematical stability of a Batch Size of 16, you set gradient_accumulation_steps = 8.
    - The model looks at 2 examples and calculates the gradients (the required weight updates).
    - Instead of applying them, it just holds that math in its head (accumulates).
    - It does this 8 times in a row.
    - Once it has looked at 16 total examples ($2 \times 8$), it combines all the math and finally updates the weights.

- Why it matters: You get the smooth, stable learning of a massive server cluster, running safely on your local desktop.

### 3. Learning Rate (learning_rate):
- What it is: The "Step Size." Once the model calculates its error, the learning rate dictates how aggressively it should change its weights to fix that error.

- How it affects Fine-Tuning:

    - Too High (e.g., 1e-2 / 0.01): The model takes massive steps, overshoots the correct answer, and the loss curve explodes into infinity. The model learns nothing.

    - Too Low (e.g., 1e-7 / 0.0000001): The model takes microscopic steps. It will take weeks to learn anything, and it might get "stuck" in a sub-optimal state.

    - The LoRA Standard: Because we are only training a tiny adapter, we can use slightly higher learning rates than full fine-tuning. 2e-4 (0.0002) or 2e-5 (0.00002) are the undisputed golden standards for QLoRA.

### 4. num_train_epochs:
- What it is: One "Epoch" means the model has looked at every single item in your dataset exactly one time.

- How it affects Fine-Tuning:

    - Unlike traditional machine learning where you might train for 50 or 100 epochs, LLMs are incredibly prone to Catastrophic Forgetting and Overfitting.

    - If you train an LLM on the same text too many times, it will start memorizing the text word-for-word and lose its ability to speak conversationally.

    - The Standard: Most LLM fine-tuning runs for just 1 to 3 Epochs. Quality of data matters far more than the number of times the model reads it.

### 5. Optimizers (optim):
- What it is: The optimizer is the algorithm that actually applies the Learning Rate to the weights.

- How it works in QLoRA:

    - AdamW: The undisputed king of modern deep learning. It dynamically adjusts the learning rate for every single parameter individually (some weights need big steps, some need small steps).

    - The Local Savior (paged_adamw_8bit or paged_adamw_32bit): Standard AdamW consumes a massive amount of VRAM because it has to store a "memory" of past gradients for every single weight. Hugging Face provides a "paged" version that actively moves memory between your GPU and your CPU RAM to prevent spikes, and an "8-bit" version that compresses the optimizer's memory footprint drastically.